## 1. Import required Libraries

In [10]:
import os
from PIL import Image
from collections import Counter
import cv2
import numpy as np
import pandas as pd

## 2. Image Resizing

In [11]:
# ===============================
# Cấu hình
# ===============================
labels_file = "image_labels.txt"
dataset_path = "dataset"  # thư mục chứa ảnh đã copy
required_size = (150, 150)

# ===============================
# 1) Nạp labels từ file
# ===============================
labels = []
if not os.path.exists(labels_file):
    raise FileNotFoundError(f"Không thấy file {labels_file}. Hãy chạy cell tạo nhãn trước.")

with open(labels_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(": ")
        if len(parts) == 2:
            fname, lbl = parts[0], int(parts[1])
            labels.append((fname, lbl))

# Map filename -> label
label_map = {fname: lbl for fname, lbl in labels}

# ===============================
# 2) Resize và thống kê theo stage
# ===============================
stage_fixed = Counter()   # số ảnh đã resize theo stage
stage_seen  = Counter()   # số ảnh hợp lệ đã duyệt theo stage
count_fixed = 0
count_unreadable = 0

print("🔍 Đang kiểm tra và resize ảnh sai kích thước...\n")

for filename in os.listdir(dataset_path):
    img_path = os.path.join(dataset_path, filename)

    # bỏ qua non-image
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    lbl = label_map.get(filename, None)
    if lbl is not None:
        stage_seen[lbl] += 1

    # ---------------------------
    # Thử mở bằng Pillow
    # ---------------------------
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            if (w, h) != required_size:
                print(f"⚠️ {filename} → {w}x{h} ❌  → resizing to 150x150")
                resized_img = img.resize(required_size, Image.LANCZOS)
                resized_img.save(img_path)
                count_fixed += 1
                if lbl is not None:
                    stage_fixed[lbl] += 1

    except (UnidentifiedImageError, OSError, ValueError):
        # ---------------------------
        # Fallback: dùng OpenCV nếu Pillow fail
        # ---------------------------
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            print(f"❌ Không thể đọc ảnh: {filename} | skipped")
            count_unreadable += 1
            continue
        h, w = img_cv.shape[:2]
        if (w, h) != required_size:
            print(f"⚠️ {filename} → {w}x{h} ❌  → resizing to 150x150 using OpenCV")
            resized_cv = cv2.resize(img_cv, required_size, interpolation=cv2.INTER_AREA)
            cv2.imwrite(img_path, resized_cv)
            count_fixed += 1
            if lbl is not None:
                stage_fixed[lbl] += 1

# ===============================
# 3) Báo cáo
# ===============================
print("\n======================")
print(f"✨ Tổng ảnh đã sửa kích thước: {count_fixed}")
print(f"❌ Tổng ảnh không đọc được: {count_unreadable}")
print("======================\n")

stage_names = [f"Stage {i+1}" for i in range(8)]
for i, name in enumerate(stage_names):
    print(f"{name}: đã thấy {stage_seen.get(i,0)} ảnh, sửa {stage_fixed.get(i,0)} ảnh")

if count_fixed == 0 and count_unreadable == 0:
    print("\n🎉 Tất cả ảnh đều có đúng kích thước 150×150!")
else:
    print("\n✨ Đã resize tất cả ảnh sai kích thước thành công!")

🔍 Đang kiểm tra và resize ảnh sai kích thước...

⚠️ Stage1_10_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_1_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_2_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_3_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_4_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_5_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_6_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_7_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_8_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_9_U2269035.jpg → 113x150 ❌  → resizing to 150x150
⚠️ Stage1_U2361325.jpg_1928.JPG → 5712x4284 ❌  → resizing to 150x150
⚠️ Stage1_U2361325.jpg_1929.JPG → 5712x4284 ❌  → resizing to 150x150
⚠️ Stage1_U2361325.jpg_1930.JPG → 5712x4284 ❌  → resizing to 150x150
⚠️ Stage1_U2361325.jpg_1931.JPG → 5712x4284 ❌  → resizing to 150x150
⚠️ Stage1_U2361325.jpg_1932.JPG → 5712x4284 ❌  → resizing to 150x150
⚠️ St

## 2. Cleanlab

In [4]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras import layers, models, callbacks
from cleanlab.filter import find_label_issues

print("✨ Extracting features using MobileNetV2...")
base_model = tf.keras.applications.MobileNetV2(weights="imagenet", include_top=False, pooling="avg")
X_images_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(X_images)
X_features = base_model.predict(X_images_preprocessed, verbose=1)

# Dữ liệu/nhãn tương ứng valid_indices
X_valid = X_features
y_valid = y_labels[valid_indices]
n_classes = 8

def build_head(input_dim, n_classes=8):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation="relu")(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

print("🔁 Building out-of-fold probabilities with StratifiedKFold + MLP head...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(X_valid), n_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_valid, y_valid), 1):
    print(f"  Fold {fold}")
    X_tr, X_va = X_valid[tr_idx], X_valid[va_idx]
    y_tr, y_va = y_valid[tr_idx], y_valid[va_idx]

    model = build_head(X_valid.shape[1], n_classes)
    es = callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_loss")
    rlrop = callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5, verbose=1)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=20,
        batch_size=64,
        callbacks=[es, rlrop],
        verbose=1
    )
    probs_va = model.predict(X_va, verbose=0)
    oof_probs[va_idx] = probs_va

print("✅ Done building out-of-fold probabilities.")

ModuleNotFoundError: No module named 'cleanlab'

In [ ]:
print("🔍 Cleanlab đang tìm các mẫu nghi ngờ gán nhãn sai...")
label_issues = find_label_issues(
    labels=y_valid,
    pred_probs=oof_probs
)
print("🔎 Cleanlab flagged:", label_issues.sum(), "potential label errors")